# Why does a robot policy fail? A guided look at one PushT run

**What this is.** A lab bench for recordings. A pretrained robot policy was run on a simple pushing task and everything it did was saved. This notebook only *re-reads* those recordings, so you can change a setting and see the result at once. Nothing here runs the robot (that is slow and is the agent's job).

**Where this is going.** The end goal of the project is to understand why policies succeed or fail at *assembly and disassembly* (putting a nut on a peg, taking it off). Assembly is hard for a few specific reasons, and PushT is a small, cheap task that has some of them. We learn to read a policy here first, then point the same tools at real assembly.

**What you will learn in this notebook.** Whether a policy that fails is *indecisive*: does it keep changing its mind about which way to go around the object? That is research question **Q-001** (assembly difficulty "A": several valid ways to approach a part).

**How to use it.** Run the cells from top to bottom. Each section says *what you are looking at*, *what to try* and *what to notice*. Before you touch a slider, write down what you expect. Being wrong is where the learning is.

## 1. The task: PushT

A round pusher (the *robot*, drawn as a dot) must push a **T-shaped block** until it lies exactly on a fixed **target outline** of the same T. The camera looks down on the table, like a 2D game.

An episode ends when the T covers the target by at least **95%** or after **300 steps**. Why this task? Two assembly problems appear in it in miniature:

- **The robot can go around the T on the left or on the right.** Both work, but half of one and half of the other does not. (Assembly: several valid ways to approach or grasp a part.)
- **The last few millimetres matter.** 94% overlap is a failure. (Assembly: the fit before insertion has to be precise.)

Outcome labels used everywhere below:

| label | meaning |
|---|---|
| `success` | T covers the target by 95% or more |
| `near_miss` | not a success, but it got to 80% or more at some point |
| `fail` | never got close |

## 2. The policy: Diffusion Policy (pretrained)

We use the **Diffusion Policy** checkpoint `lerobot/diffusion_pusht`, trained by others and published through the LeRobot library. We did not train it. Its model card reports **65.4%** success on PushT, so failures are common, which is what we want to study.

How it decides what to do, in four sentences:

1. It looks at the camera image (96×96 pixels) and the robot's own position.
2. It starts from random noise and *denoises* it, in 100 passes, into a plan: the next stretch of movements.
3. It carries out the first **8 steps** of that plan, then looks again and **re-plans**. That 8-step block is called a *chunk* below.
4. Because every plan starts from fresh random noise, the same situation can lead to a plan that goes left one time and right the next. That is how it can represent "left or right", and also how it could flip-flop between them.

Everything saved about each run (which checkpoint, which code, which seeds) is listed in section 4.

## 3. The question and how we measure it

**Q-001:** *When the policy fails, did it commit to one way around the T, or did it keep switching?*

To measure "which way around", we track where the robot is **around the T**, as an angle, from the T's own point of view (so pushing the T along does not count as moving around it). The angle line goes **up** when the robot circles counter-clockwise and **down** when it circles clockwise. Three words follow from that:

- **Swing**: during one chunk (one 8-step plan) the robot turned by at least some amount, the *min swing* (default 0.5 rad, about 29°). Smaller turns are treated as jitter.
- **Switch**: a swing in the *opposite* direction to the previous swing. A switch is the robot changing its mind.
- **Chunk**: the 8 steps between re-plans. Changing the chunk changes how finely we look.

Two settings decide what gets counted: **min swing** and **chunk**. They are choices, not facts, so a good result should survive moving them. Every figure below states the values it used.

## 4. Load the data

`RUN` is the folder of one recorded run. Point it at another folder in `runs/` to look at that run instead. Loading reads each episode's positions over time (small). The network activations stay on disk until you ask for them in section 8.

In [4]:
from plens import explore, explore_ui

RUN = "../runs/20261021-153458_sides"   # one run folder; change this to look at a different run
run = explore.load(RUN)

print(f"{len(run.episodes)} episodes loaded from {run.run_dir.resolve()}")
print("outcomes:", {label: len(eps) for label, eps in run.by_label().items()})

10 episodes loaded from /Users/imanm/policy lens/runs/20261021-153458_sides
outcomes: {'success': 6, 'fail': 2, 'near_miss': 2}


### Where this data came from

The table below is read directly from the run's `meta.json`, which was written when the run started and finished. It answers: *which files hold the data, which policy and weights produced it, what command was run, with which code version, and with which settings.* If a finding is ever questioned, this is where you start. Check that **code version** is not marked as having uncommitted changes.

In [ ]:
explore_ui.provenance_panel(run)

## 5. A toy first: learn what the measurement counts

**What you are looking at.** A made-up robot, no real data. In the **angles** box you type a list of places for it to go around the T, one after another. The numbers are angles in radians: `0` is where it starts, `1.5` is about a quarter turn counter-clockwise, a negative number is clockwise. The robot moves smoothly from each number to the next, and the plot shows its path (up = counter-clockwise, down = clockwise).

An **open circle** marks a *switch*: a point where the measurement says "the robot turned round and went the other way". The plot title compares two things: how many reversals **you drew**, and how many switches the measurement **counted**. If they differ, the measurement got it wrong, and that is the point of this section.

**Try these, in order:**
1. Type `0, 1.5, 0, 1.5`. The robot goes counter-clockwise (0 to 1.5), turns back (1.5 to 0), then turns again (0 to 1.5). That is 2 reversals, and you should see 2 open circles.
2. Now raise **min swing** to 1.5. The measurement only counts a turn if the robot rotates at least that much within a single re-plan (the thin grey lines are the re-plans). No re-plan is that big, so it counts 0: real reversals were missed.
3. Set min swing back to 0.5 and slowly raise **jitter** from 0 to 0.3. Jitter adds random wobble, like a real robot never moving perfectly smoothly. Watch the "counted" number: if it goes above the 2 you drew, the measurement is reporting changes of mind that never happened.
4. Type `0, 0.8, 0, 0.8` (the same up-down-up, but smaller) and set **chunk** to 4, then 8, then 16. The chunk is how many steps we look at together when deciding whether the robot turned. At 4 and 8 the robot never turns by 0.5 within one chunk, so 0 are counted. At 16 it does, so 2 are counted. The robot moved identically each time; only the way we looked changed.

**Notice.** The count depends on the settings as much as on the robot. It can miss real reversals (steps 2 and 4) and invent false ones (step 3). Keep this in mind when you read the real data: a number like "4.5 switches" always means "4.5 under these settings".

In [5]:
explore_ui.toy_panel()

## 6. Real paths, and does the finding survive a different threshold?

**What you are looking at.**
- **Left:** one line per episode, colour = outcome (green success, orange near miss, red fail). The number at the right end of each line is the episode number. The line shows the robot's angle around the T over time. **Open dots** are switches.
- **Right:** the average number of switches per episode for each outcome, at every possible min swing. The dashed line is where your slider currently is.
- **Table below:** the same numbers for each single episode.

Where a line starts means little (it depends on where the robot started). Look at the *shape*: long climbs or falls mean steady circling, zig-zags mean changing direction.

**Try:**
1. Drag **min swing** from 0.2 to 1.0 and watch the right-hand lines. A stronger finding keeps the red line above the green one.
2. Change **chunk**, and tick **show re-plan lines**.
3. Find the two failure lines on the left. Do they zig-zag, or do they go one way and just get stuck?

**Notice.** In the first run (10 episodes) there were only 2 failures (finding F-002). A gap between 2 failures and 6 successes is a hint, not a result; use this section to see *how fragile* it is.

**Predict before you drag:** failures switch around step ______ . At a min swing of 1.0 rad, the gap between fail and success will be ______ .

In [6]:
explore_ui.threshold_panel(run)

## 7. Watch an episode next to its angle line

**What you are looking at.** One episode. **Left:** what the robot did (the recorded video, if one was saved for that episode; otherwise a simple drawing). **Right:** the angle line, with a black cursor at the current step. Drag **step** to scrub through the episode, or press ▶.

**Try:** pick one success, one near miss and one failure. For the failure, scrub to each open dot and look at the left picture: what was the robot doing when the line turned around?

**Notice.** Numbers like "4.5 switches" hide what happened physically. Does a switch look like a real change of plan, or like the robot nudging the T?

*If playing is jerky,* that is the notebook redrawing each frame on demand. The next cell makes a smooth MP4 instead.

In [7]:
explore_ui.replay_panel(run)

### A smooth version: make an MP4

Pick an episode and a min swing, press **Make MP4**, and wait about a minute. The video plays here and is also saved in an `exports/` folder next to this notebook (not inside the run, because runs are never modified). Each frame of the video has a footer naming the files and settings it came from.

In [ ]:
explore_ui.mp4_panel(run)

## 8. Inside the network: do the activations change when the robot changes its mind?

**What you are looking at.** While the robot ran, we also recorded what happens *inside* the policy, at two points (one number list per re-plan):

- **what it sees** (`rgb_encoder`): the image encoder, which turns the camera picture into numbers;
- **what it plans** (`unet.mid_modules.1`): the middle of the denoiser, where the plan is being formed.

These are long lists of numbers, so two views make them readable:
- **Left:** the layer's state squeezed onto two summary axes (called PCA; it keeps as much of the variation as two axes can). Each dot is one re-plan, dark to light is time, the ring is "now". The axes have no meaning of their own: look at *how far and how fast* the path moves, not the coordinates.
- **Middle:** how much the layer changed from one re-plan to the next. A high point means "this re-plan was very different from the last".
- **Right:** the same angle line as before, for reference.

**Try:** pick a failure, then compare the middle plot with its switches on the right. Switch layers and compare.

**Notice.** Be careful: the layer also changes when the *scene* changes (the T moved, the robot is somewhere new), not only when the robot changes its mind. A jump next to a switch is a lead, not proof. This is the first step towards Q-003 and Q-004 (can we read the state of the task from the activations?).

**Predict:** the activations jump at the re-plan where the robot switches side. ( yes / no / only for one of the two layers )

In [8]:
explore_ui.activation_panel(run)

## What surprised me

-
-

## What I would ask the agent to do next

Bring this list to the agent. Good requests describe a question, not code, for example: *"run 30 more episodes so we have 10 or more failures"*, or *"record the same layers for the near misses only"*.